
# Planet → Lake Mask Pipeline (Single Lakes Shapefile → Raster Labels)

This notebook creates binary lake masks (**1 = lake, 0 = not lake**) aligned to your **PlanetScope SR (8-band)** GeoTIFFs.  
It uses a **single lakes Shapefile** that contains *many polygons* (one polygon per lake boundary).  
If a **UDM2** file is provided, pixels flagged as unusable become **255 (ignore)** for training.

**You provide:**
- SR image(s): e.g. `20200906_050946_11_2235_3B_AnalyticMS_SR_8b_clip_reproject_file_format.tif`
- UDM2 image(s): e.g. `20200906_050946_11_2235_3B_udm2_clip_reproject_file_format.tif` (optional)
- A **single** lakes Shapefile (`.shp`) with polygons for all lakes.

**Output:**
- A label GeoTIFF per SR image with values: `1` (lake), `0` (non-lake), `255` (ignore via UDM2).


## 0) Requirements

In [ ]:

# If running locally and you need to install packages, uncomment:
# %pip install geopandas rasterio shapely pyproj tqdm numpy fiona


## 1) Imports

In [1]:

from pathlib import Path
import numpy as np
import rasterio
from rasterio.features import rasterize
from rasterio.warp import reproject, Resampling
import geopandas as gpd
from shapely.geometry import box
from shapely.ops import unary_union
from tqdm import tqdm


## 2) Configuration

In [2]:

# --- EDIT THESE PATHS ---
# Folder holding your SR and UDM2 GeoTIFFs (they can be in the same folder)
DATA_DIR = Path("D:\\Thesis\\glacial_lake_detection_thesis\\Training\\3 Preparing train data\\rgb_outputs\\complete_chips_out")   # change to your folder

# A **single** lakes Shapefile containing polygons for all lakes
LAKES_SHP = Path("HKH-PK.zip")  # <— change to your lakes .shp path

# Example single-file run (edit to your actual filenames if you want to run only one pair)
SR_TIF = DATA_DIR / "20200906_050946_11_2235_3B_AnalyticMS_SR_8b_clip_reproject_file_format.tif"
UDM2_TIF = None  # set to None if not using UDM2

# Batch mode matching patterns:
SR_MATCH_TOKEN = "_AnalyticMS_SR_8b_"

def sr_to_udm2(sr_path: Path) -> Path:
    """Best-effort mapping from SR filename to UDM2 filename."""
    return sr_path.with_name(sr_path.name.replace("_AnalyticMS_SR_8b_", "_udm2_"))

# Output folder
OUT_DIR = Path("./rgb_outputs/labels_out")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Label schema
LAKE_VAL = 1
NON_LAKE_VAL = 0
IGNORE_VAL = 255  # used where UDM2 marks pixel as bad

FILL_VALUE = NON_LAKE_VAL

print("Configured paths:")
print("DATA_DIR =", DATA_DIR.resolve())
print("LAKES_SHP =", LAKES_SHP.resolve())
print("OUT_DIR =", OUT_DIR.resolve())


Configured paths:
DATA_DIR = D:\Thesis\glacial_lake_detection_thesis\Training\3 Preparing train data\rgb_outputs\complete_chips_out
LAKES_SHP = D:\Thesis\glacial_lake_detection_thesis\Training\3 Preparing train data\HKH-PK.zip
OUT_DIR = D:\Thesis\glacial_lake_detection_thesis\Training\3 Preparing train data\rgb_outputs\labels_out


## 3) Load lake polygons from the single Shapefile and align CRS

In [3]:

def load_lake_polygons_from_single_shapefile(sr_dataset, lakes_shp: Path):
    """
    Read a single lakes Shapefile with many polygons (one per lake), reproject to the SR CRS,
    keep only polygons intersecting the SR bounds, and return a list of geometries for rasterization.
    We union overlapping polygons to avoid double rasterization.
    """
    if not lakes_shp.exists():
        raise FileNotFoundError(f"Lakes shapefile not found: {lakes_shp}")

    gdf = gpd.read_file(lakes_shp)
    if gdf.empty:
        print(f"WARNING: {lakes_shp} is empty.")
        return []
    if gdf.crs is None:
        raise ValueError(f"{lakes_shp} has no CRS defined. Please define it before using this notebook.")

    sr_crs = sr_dataset.crs
    sr_bounds_poly = box(*sr_dataset.bounds)

    # Reproject to SR CRS
    gdf_sr = gdf.to_crs(sr_crs)
    # Keep only polygonal geometries
    gdf_sr = gdf_sr[gdf_sr.geometry.type.isin(["Polygon", "MultiPolygon"])]
    # Intersect with raster bounds
    gdf_sr = gdf_sr[gdf_sr.geometry.intersects(sr_bounds_poly)]
    if gdf_sr.empty:
        return []

    # Union overlapping polygons to avoid duplicates
    try:
        unioned = unary_union([geom for geom in gdf_sr.geometry if geom is not None and not geom.is_empty])
    except Exception as e:
        print(sr_dataset)

    if unioned.is_empty:
        return []

    # Return as list of (multi)polygons
    if unioned.geom_type in ("Polygon", "MultiPolygon"):
        return [unioned]
    # GeometryCollection fallback
    return [g for g in getattr(unioned, "geoms", []) if g.geom_type in ("Polygon", "MultiPolygon")]


## 4) UDM2 → valid/invalid mask (heuristics)

In [11]:

def udm2_to_ignore_mask(udm2_path: Path, sr_dataset) -> np.ndarray:
    """Convert UDM2 to boolean ignore mask aligned with SR (True = ignore)."""
    with rasterio.open(udm2_path) as udm2:
        same_grid = (udm2.transform == sr_dataset.transform) and (udm2.width == sr_dataset.width) and (udm2.height == sr_dataset.height) and (udm2.crs == sr_dataset.crs)
        if same_grid:
            arr = udm2.read()
        else:
            arr = udm2.read()
            out = np.zeros((udm2.count, sr_dataset.height, sr_dataset.width), dtype=arr.dtype)
            for b in range(udm2.count):
                reproject(
                    source=arr[b],
                    destination=out[b],
                    src_transform=udm2.transform,
                    src_crs=udm2.crs,
                    dst_transform=sr_dataset.transform,
                    dst_crs=sr_dataset.crs,
                    resampling=Resampling.nearest
                )
            arr = out

        if arr.shape[0] == 1:
            band = arr[0]
            return (band == 0)  # assume 1=valid, 0=invalid
        clear = arr[0]
        others = arr[1:]
        any_flag = np.any(others > 0, axis=0) if others.size else False
        return (clear == 0) | any_flag


## 5) Build the label mask aligned to SR using the single lakes Shapefile

In [4]:
import csv
def build_label_mask_for_sr(sr_path: Path, lakes_shp: Path, out_path: Path, udm2_path: Path | None = None,
                            lake_val: int = 1, non_lake_val: int = 0, ignore_val: int = 255):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(sr_path) as sr:
        lake_polys = load_lake_polygons_from_single_shapefile(sr, lakes_shp)

        # Base mask: non-lake
        mask = np.full((sr.height, sr.width), fill_value=non_lake_val, dtype=np.uint8)

        if lake_polys:
            # Rasterize polygons → value=lake
            lake_raster = rasterize(
                [(geom, lake_val) for geom in lake_polys],
                out_shape=(sr.height, sr.width),
                transform=sr.transform,
                fill=non_lake_val,
                dtype=np.uint8
            )
            mask = lake_raster.astype(np.uint8)
        #else:
            #print(f"No lake polygons intersect {sr_path.name}; writing all {non_lake_val}.")

        #if udm2_path is not None and Path(udm2_path).exists():
            #ignore_mask = udm2_to_ignore_mask(Path(udm2_path), sr)
            #mask = np.where(ignore_mask, ignore_val, mask).astype(np.uint8)

        profile = sr.profile.copy()
        profile.update({
            "count": 1,
            "dtype": "uint8",
            "nodata": ignore_val
        })
        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(mask, 1)

        #n_ignore = int((mask == ignore_val).sum())
        n_lake = int((mask == lake_val).sum())
        n_nonlake = int((mask == non_lake_val).sum())
        #print(f"Wrote {out_path.name} → shape={mask.shape}, lake={n_lake:,}, nonlake={n_nonlake:,}")
                # Append run stats to a CSV (one row per call)
        csv_path = out_path.parent / "label_mask_stats.csv"
        csv_path.parent.mkdir(parents=True, exist_ok=True)
        write_header = not csv_path.exists()
        with csv_path.open("a", newline="") as f:
            writer = csv.writer(f)
            if write_header:
                writer.writerow([
                    "sr_path", "out_mask", "height", "width",
                    "lake_val", "non_lake_val", "ignore_val",
                    "n_lake", "n_nonlake"
                ])
            writer.writerow([
                str(sr_path), out_path.name, mask.shape[0], mask.shape[1],
                int(lake_val), int(non_lake_val), int(ignore_val),
                n_lake, n_nonlake
            ])



## 6) Run → Single tile

In [21]:

if SR_TIF.exists():
    out_single = OUT_DIR / (SR_TIF.stem + "_label.tif")
    use_udm2 = UDM2_TIF if (UDM2_TIF is not None and UDM2_TIF.exists()) else None
    build_label_mask_for_sr(SR_TIF, LAKES_SHP, out_single, use_udm2,
                            lake_val=LAKE_VAL, non_lake_val=NON_LAKE_VAL, ignore_val=IGNORE_VAL)
else:
    print("Edit SR_TIF/UDM2_TIF/paths above to run the single-tile example.")


[<MULTIPOLYGON (((71.602 35.589, 71.602 35.589, 71.602 35.589, 71.602 35.589,...>]
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
Wrote 20200906_050946_11_2235_3B_AnalyticMS_SR_8b_clip_reproject_file_format_label.tif → shape=(678, 752), lake=610, nonlake=509,246


## 7) Batch mode → Process all SR tiles in a folder

In [22]:

sr_files = sorted([p for p in DATA_DIR.glob("*.tif")])
print(f"Found {len(sr_files)} SR tiles")

for sr_path in tqdm(sr_files):
    out_label = OUT_DIR / (sr_path.stem + "_label.tif")
    udm2_guess = sr_to_udm2(sr_path)
    use_udm2 = udm2_guess if udm2_guess.exists() else None
    build_label_mask_for_sr(sr_path, LAKES_SHP, out_label, use_udm2,
                            lake_val=LAKE_VAL, non_lake_val=NON_LAKE_VAL, ignore_val=IGNORE_VAL)


Found 1529 SR tiles


 71%|███████▏  | 1091/1529 [03:45<01:30,  4.83it/s]

<open DatasetReader name='D:\Thesis\glacial_lake_detection_thesis\Training\3 Preparing train data\rgb_outputs\complete_chips_out\20200922_055934_31_241c_3B_Visual_clip_reproject_file_format_r2863_c7362_512px.tif' mode='r'>


UnboundLocalError: cannot access local variable 'unioned' where it is not associated with a value

## 8) (Optional) Build a CSV index of SR/UDM2/Label paths

In [ ]:

import csv

index_csv = OUT_DIR / "dataset_index_single_shp.csv"
rows = []
for lbl in sorted(OUT_DIR.glob("*_label.tif")):
    sr_name = lbl.name.replace("_label.tif", ".tif")
    candidate_sr = DATA_DIR / sr_name
    candidate_udm2 = sr_to_udm2(candidate_sr)
    rows.append({
        "sr_path": str(candidate_sr.resolve() if candidate_sr.exists() else ""),
        "udm2_path": str(candidate_udm2.resolve() if candidate_udm2.exists() else ""),
        "label_path": str(lbl.resolve())
    })

with open(index_csv, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["sr_path", "udm2_path", "label_path"])
    w.writeheader()
    w.writerows(rows)

print("Wrote", index_csv.resolve())



## Notes & Tips

- **Single Shapefile**: Set `LAKES_SHP` to your lakes `.shp` file containing **all** lake polygons.
- **CRS matters**: The shapefile is reprojected to the SR tile's CRS before rasterization. Ensure the shapefile CRS is defined.
- **Union of polygons**: Overlapping polygons are **unioned** to avoid duplicates. If you prefer not to union (e.g., overlapping lakes are meaningful), rasterize `gdf_sr.geometry` directly.
- **UDM2 heuristics**: If your UDM2 encodes flags differently, adjust `udm2_to_ignore_mask`.
- **Training**: The label band is `uint8` with `nodata=255`. Classes: `0=non-lake`, `1=lake`. Ignore is `255`.
